# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² dataset using the `mlcroissant` library. The focus is on hands-on exploration of records, fields, and columns using their Croissant `@id` identifiers.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
ds = mlc.Dataset(croissant_url)
metadata = ds.metadata
print(f"{metadata.name}: {metadata.description}\n")
print("Published on:", metadata.date_published)
print("Version:", metadata.version)
print("License:", metadata.license)


## 2. Data Overview
Review available record sets, fields, and their IDs.

We use the Croissant `@id` for all entities. Let's print available record sets (tables), along with their fields and columns.

In [ ]:
# List all record sets and display their @id, fields, and sample columns
for rs in metadata.record_sets:
    print(f"RecordSet: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print("  Fields and their @id:" )
    for fld in rs.fields:
        print(f"    - {fld.name} (@id: {fld.id}, type: {fld.data_type})")
        if hasattr(fld, 'column') and fld.column is not None:
            # Some fields may have single or multiple columns
            if isinstance(fld.column, list):
                for col in fld.column:
                    print(f"      column @id: {col.id}")
            else:
                print(f"      column @id: {fld.column.id}")
    print("---\n")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 

Below we:
- Collect all available record set `@id`s.
- Load each record set's data into a pandas DataFrame using `mlcroissant.Dataset.records()`.
- Display the DataFrame columns and a preview for one record set.


In [ ]:
# Collect all record_set @id values
record_sets_ids = [rs.id for rs in metadata.record_sets]
print("Available RecordSet @id values:", record_sets_ids)

dataframes = {}
for record_set_id in record_sets_ids:
    # Each record is a dict where keys are field @id
    records = list(ds.records(record_set=record_set_id))
    if records:  # Only create DF if data exists
        dataframes[record_set_id] = pd.DataFrame(records)

# If at least one DataFrame loaded, choose the first for overview
example_record_set = record_sets_ids[0]
print(f"\nColumns for record set '{example_record_set}':")
print(dataframes[example_record_set].columns.tolist())
print("\nPreview of first 5 rows:")
dataframes[example_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Let's process the data for basic analysis:
- Choose a numeric field by its `@id`.
- Filter out records below a threshold.
- Standardize that numeric field (mean=0, std=1).
- Optionally, group by a categorical field's `@id`.


In [ ]:
# Replace the following values with the actual field @ids from the previous overview as needed
# For this dataset, let's pick likely numeric and group fields based on field names
record_set_id = example_record_set

# Try to pick a numeric field for EDA
df = dataframes[record_set_id]
numeric_field_id = None
group_field_id = None
# Try to auto-detect (field @id containing 'Abundance', 'MW', 'Score', 'Count')
for col in df.columns:
    if any(x in col.lower() for x in ['abundance', 'mw', 'score', 'count', 'coverage']):
        numeric_field_id = col
        break
for col in df.columns:
    if any(x in col.lower() for x in ['sample', 'class', 'group', 'category', 'type', 'protein']):
        if col != numeric_field_id:
            group_field_id = col
            break

print(f"Selected numeric field: {numeric_field_id}")
print(f"Selected group field: {group_field_id}")

# Make sure field exists and convert to numeric
if numeric_field_id in df.columns:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean()  # Example: use mean as threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} rows\n")

    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Example normalized values for {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Optionally group
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped by '{group_field_id}':")
        print(grouped_df.head())
else:
    print('No suitable numeric field found for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    sns.set(style="whitegrid")
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:

- Load a Croissant schema-based dataset via `mlcroissant`
- List all available record sets and fields via their `@id`
- Extract data dynamically for each record set
- Perform basic exploratory data analysis and visualizations using field `@id`s

By referencing all dataset entities (record sets, fields, columns) by their Croissant `@id` values, workflows remain robust, reproducible, and interoperable across various dataset versions.